## Import Required Libraries

In [ ]:
# Install required packages
!pip install pandas numpy scikit-learn xgboost matplotlib seaborn plotly imbalanced-learn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Machine Learning Libraries
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score, 
    precision_score, recall_score, f1_score, roc_auc_score,
    roc_curve, precision_recall_curve
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import GradientBoostingClassifier
import xgboost as xgb
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

import pickle
import warnings
warnings.filterwarnings('ignore')

# Set style for plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Random seed for reproducibility
np.random.seed(42)

## Data Loading and Initial Exploration

In [ ]:
# Load the fetal health dataset
df = pd.read_csv('/home/user/HEC/fetal_health.csv')

print("Dataset shape:", df.shape)
print("\nFirst 5 rows:")
df.head()

In [ ]:
# Basic dataset information
print("Dataset Info:")
print(df.info())

print("\nDataset Description:")
df.describe()

In [ ]:
# Check for missing values
print("Missing Values:")
print(df.isnull().sum())

print("\nTarget Variable Distribution:")
print(df['fetal_health'].value_counts().sort_index())

# Map the target values to meaningful labels
health_labels = {1: 'Normal', 2: 'Suspect', 3: 'Pathological'}
df['health_status'] = df['fetal_health'].map(health_labels)

print("\nHealth Status Distribution:")
print(df['health_status'].value_counts())

## Exploratory Data Analysis

In [ ]:
# Target distribution visualization
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Count plot
df['health_status'].value_counts().plot(kind='bar', ax=axes[0], color=['green', 'orange', 'red'])
axes[0].set_title('Fetal Health Status Distribution')
axes[0].set_xlabel('Health Status')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Pie chart
df['health_status'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%', 
                                        colors=['green', 'orange', 'red'])
axes[1].set_title('Fetal Health Status Proportion')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation matrix
plt.figure(figsize=(20, 16))
correlation_matrix = df.select_dtypes(include=[np.number]).corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of key features by health status
key_features = ['baseline value', 'accelerations', 'fetal_movement', 'uterine_contractions', 
                'severe_decelerations', 'abnormal_short_term_variability']

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.ravel()

for i, feature in enumerate(key_features):
    for health_status in df['health_status'].unique():
        subset = df[df['health_status'] == health_status][feature]
        axes[i].hist(subset, alpha=0.7, label=health_status, bins=30)
    
    axes[i].set_title(f'Distribution of {feature}')
    axes[i].set_xlabel(feature)
    axes[i].set_ylabel('Frequency')
    axes[i].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Box plots for key features
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.ravel()

for i, feature in enumerate(key_features):
    df.boxplot(column=feature, by='health_status', ax=axes[i])
    axes[i].set_title(f'Box Plot: {feature} by Health Status')
    axes[i].set_xlabel('Health Status')
    axes[i].set_ylabel(feature)

plt.suptitle('')  # Remove automatic title
plt.tight_layout()
plt.show()

## Feature Engineering and Preprocessing

In [ ]:
# Create new features based on domain knowledge
df_processed = df.copy()

# Risk indicators
df_processed['total_decelerations'] = (df_processed['light_decelerations'] + 
                                     df_processed['severe_decelerations'] + 
                                     df_processed['prolongued_decelerations'])

df_processed['deceleration_risk'] = (df_processed['severe_decelerations'] > 0) | \
                                  (df_processed['prolongued_decelerations'] > 0)

# Variability indicators
df_processed['variability_ratio'] = (df_processed['mean_value_of_short_term_variability'] / 
                                   (df_processed['mean_value_of_long_term_variability'] + 1e-6))

# Histogram features
df_processed['histogram_range'] = df_processed['histogram_max'] - df_processed['histogram_min']
df_processed['histogram_asymmetry'] = df_processed['histogram_mean'] - df_processed['histogram_median']

# Baseline abnormality
df_processed['baseline_abnormal'] = ((df_processed['baseline value'] < 110) | 
                                   (df_processed['baseline value'] > 160)).astype(int)

print("New features created:")
new_features = ['total_decelerations', 'deceleration_risk', 'variability_ratio', 
               'histogram_range', 'histogram_asymmetry', 'baseline_abnormal']
print(new_features)

# Show some statistics of new features
print("\nNew features statistics:")
df_processed[new_features].describe()

In [ ]:
# Prepare features and target
# Remove non-numeric and target columns
feature_cols = [col for col in df_processed.columns if col not in ['fetal_health', 'health_status']]
X = df_processed[feature_cols]
y = df_processed['fetal_health'] - 1  # Convert to 0, 1, 2 for sklearn compatibility

print(f"Feature matrix shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeatures: {list(X.columns)}")
print(f"\nTarget distribution: {np.bincount(y)}")

In [ ]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Further split training into train and validation
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

print(f"Training set: {X_train.shape}, {y_train.shape}")
print(f"Validation set: {X_val.shape}, {y_val.shape}")
print(f"Test set: {X_test.shape}, {y_test.shape}")

# Check class distribution in splits
print(f"\nTrain class distribution: {np.bincount(y_train)}")
print(f"Validation class distribution: {np.bincount(y_val)}")
print(f"Test class distribution: {np.bincount(y_test)}")

In [ ]:
# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Feature scaling completed.")
print(f"Original feature range example - baseline value: [{X_train['baseline value'].min():.2f}, {X_train['baseline value'].max():.2f}]")
print(f"Scaled feature range example: [{X_train_scaled[:, 0].min():.2f}, {X_train_scaled[:, 0].max():.2f}]")

## Model Training and Evaluation

In [ ]:
# Define evaluation function
def evaluate_model(y_true, y_pred, y_pred_proba=None, model_name="Model"):
    """
    Comprehensive model evaluation function
    """
    print(f"\n=== {model_name} Performance ===")
    
    # Basic metrics
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted')
    recall = recall_score(y_true, y_pred, average='weighted')
    f1 = f1_score(y_true, y_pred, average='weighted')
    
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-Score: {f1:.4f}")
    
    # Classification report
    print("\nClassification Report:")
    target_names = ['Normal', 'Suspect', 'Pathological']
    print(classification_report(y_true, y_pred, target_names=target_names))
    
    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=target_names, yticklabels=target_names)
    plt.title(f'{model_name} Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

In [ ]:
# Model 1: Random Forest
print("Training Random Forest Classifier...")
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    class_weight='balanced'
)

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_val)
rf_pred_proba = rf_model.predict_proba(X_val)

rf_metrics = evaluate_model(y_val, rf_pred, rf_pred_proba, "Random Forest")

In [ ]:
# Model 2: XGBoost (Main model based on the reference)
print("Training XGBoost Classifier...")

# Calculate class weights for imbalanced dataset
class_weights = len(y_train) / (3 * np.bincount(y_train))
sample_weights = np.array([class_weights[i] for i in y_train])

xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='mlogloss'
)

xgb_model.fit(X_train, y_train, sample_weight=sample_weights)
xgb_pred = xgb_model.predict(X_val)
xgb_pred_proba = xgb_model.predict_proba(X_val)

xgb_metrics = evaluate_model(y_val, xgb_pred, xgb_pred_proba, "XGBoost")

In [ ]:
# Model 3: Logistic Regression with SMOTE
print("Training Logistic Regression with SMOTE...")

# Apply SMOTE for handling imbalanced data
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42,
    multi_class='multinomial',
    solver='lbfgs'
)

lr_model.fit(X_train_smote, y_train_smote)
lr_pred = lr_model.predict(X_val_scaled)
lr_pred_proba = lr_model.predict_proba(X_val_scaled)

lr_metrics = evaluate_model(y_val, lr_pred, lr_pred_proba, "Logistic Regression (SMOTE)")

In [ ]:
# Model 4: Gradient Boosting
print("Training Gradient Boosting Classifier...")

gb_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    random_state=42
)

gb_model.fit(X_train, y_train)
gb_pred = gb_model.predict(X_val)
gb_pred_proba = gb_model.predict_proba(X_val)

gb_metrics = evaluate_model(y_val, gb_pred, gb_pred_proba, "Gradient Boosting")

In [ ]:
# Compare all models
models_comparison = pd.DataFrame({
    'Model': ['Random Forest', 'XGBoost', 'Logistic Regression (SMOTE)', 'Gradient Boosting'],
    'Accuracy': [rf_metrics['accuracy'], xgb_metrics['accuracy'], 
                lr_metrics['accuracy'], gb_metrics['accuracy']],
    'Precision': [rf_metrics['precision'], xgb_metrics['precision'], 
                 lr_metrics['precision'], gb_metrics['precision']],
    'Recall': [rf_metrics['recall'], xgb_metrics['recall'], 
              lr_metrics['recall'], gb_metrics['recall']],
    'F1-Score': [rf_metrics['f1'], xgb_metrics['f1'], 
                lr_metrics['f1'], gb_metrics['f1']]
})

print("\n=== Model Comparison ===")
print(models_comparison.round(4))

# Visualize model comparison
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
colors = ['skyblue', 'orange', 'lightgreen', 'pink']

for i, metric in enumerate(metrics):
    ax = axes[i//2, i%2]
    bars = ax.bar(models_comparison['Model'], models_comparison[metric], color=colors)
    ax.set_title(f'{metric} Comparison')
    ax.set_ylabel(metric)
    ax.tick_params(axis='x', rotation=45)
    
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

## Hyperparameter Tuning for Best Model

In [ ]:
# Select the best performing model for hyperparameter tuning
best_model_idx = models_comparison['F1-Score'].idxmax()
best_model_name = models_comparison.loc[best_model_idx, 'Model']
print(f"Best performing model: {best_model_name}")

# Hyperparameter tuning for XGBoost (assuming it's often the best)
print("\nPerforming hyperparameter tuning for XGBoost...")

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [4, 6, 8],
    'learning_rate': [0.05, 0.1, 0.15],
    'subsample': [0.8, 0.9, 1.0],
    'colsample_bytree': [0.8, 0.9, 1.0]
}

# Use a smaller grid for demo purposes
param_grid_small = {
    'n_estimators': [100, 200],
    'max_depth': [4, 6],
    'learning_rate': [0.1, 0.15],
}

xgb_tuned = xgb.XGBClassifier(random_state=42, eval_metric='mlogloss')

# Use stratified k-fold for cross-validation
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=xgb_tuned,
    param_grid=param_grid_small,
    cv=cv,
    scoring='f1_weighted',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best cross-validation score: {grid_search.best_score_:.4f}")

In [ ]:
# Train the final model with best parameters
final_model = grid_search.best_estimator_
final_pred = final_model.predict(X_val)
final_pred_proba = final_model.predict_proba(X_val)

final_metrics = evaluate_model(y_val, final_pred, final_pred_proba, "Tuned XGBoost")

## Feature Importance Analysis

In [ ]:
# Feature importance from the best model
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': final_model.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 15 Most Important Features:")
print(feature_importance.head(15))

# Visualize feature importance
plt.figure(figsize=(12, 8))
top_features = feature_importance.head(15)
sns.barplot(data=top_features, y='feature', x='importance', palette='viridis')
plt.title('Top 15 Feature Importance (XGBoost)')
plt.xlabel('Importance Score')
plt.ylabel('Features')
plt.tight_layout()
plt.show()

## Final Model Testing

In [ ]:
# Final evaluation on test set
test_pred = final_model.predict(X_test)
test_pred_proba = final_model.predict_proba(X_test)

test_metrics = evaluate_model(y_test, test_pred, test_pred_proba, "Final Model (Test Set)")

print("\n=== Final Model Performance Summary ===")
print(f"Test Accuracy: {test_metrics['accuracy']:.4f}")
print(f"Test F1-Score: {test_metrics['f1']:.4f}")
print(f"Test Precision: {test_metrics['precision']:.4f}")
print(f"Test Recall: {test_metrics['recall']:.4f}")

## Risk Analysis and Predictions

In [ ]:
# Risk-aware prediction function
def risk_aware_prediction(model, X_sample, threshold_suspect=0.3, threshold_pathological=0.7):
    """
    Make risk-aware predictions with confidence thresholds
    """
    probabilities = model.predict_proba(X_sample)
    predictions = []
    
    for prob in probabilities:
        # prob = [P(Normal), P(Suspect), P(Pathological)]
        if prob[2] > threshold_pathological:  # High confidence pathological
            predictions.append((2, 'High Risk - Pathological', prob[2]))
        elif prob[1] > threshold_suspect or prob[2] > 0.2:  # Suspect or some pathological risk
            predictions.append((1, 'Medium Risk - Suspect', max(prob[1], prob[2])))
        else:  # Normal
            predictions.append((0, 'Low Risk - Normal', prob[0]))
    
    return predictions

# Test risk-aware predictions on a few samples
sample_indices = [0, 50, 100, 150, 200]
X_samples = X_test.iloc[sample_indices]
y_samples = y_test.iloc[sample_indices]

risk_predictions = risk_aware_prediction(final_model, X_samples)

print("\n=== Risk-Aware Prediction Examples ===")
health_labels_reverse = {0: 'Normal', 1: 'Suspect', 2: 'Pathological'}

for i, (pred_class, risk_level, confidence) in enumerate(risk_predictions):
    actual_class = y_samples.iloc[i]
    print(f"Sample {sample_indices[i]}:")
    print(f"  Actual: {health_labels_reverse[actual_class]}")
    print(f"  Predicted: {risk_level} (Confidence: {confidence:.3f})")
    print(f"  Match: {'✓' if pred_class == actual_class else '✗'}")
    print()

## Model Saving and Deployment Preparation

In [ ]:
# Save the trained model and preprocessing components
import pickle
import joblib
from datetime import datetime

# Create model artifacts
model_artifacts = {
    'model': final_model,
    'scaler': scaler,
    'feature_columns': list(X.columns),
    'target_labels': {0: 'Normal', 1: 'Suspect', 2: 'Pathological'},
    'model_metrics': test_metrics,
    'feature_importance': feature_importance.to_dict(),
    'training_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
}

# Save using pickle
model_filename = 'fetal_health_model.pkl'
with open(model_filename, 'wb') as f:
    pickle.dump(model_artifacts, f)

print(f"Model saved as: {model_filename}")

# Save just the XGBoost model in native format for deployment
final_model.save_model('fetal_health_xgboost.json')
print("XGBoost model saved as: fetal_health_xgboost.json")

# Create a simple prediction function for deployment
def predict_fetal_health(model_path, input_features):
    """
    Load model and make predictions on new data
    
    Args:
        model_path: Path to the saved model file
        input_features: Dictionary or DataFrame with feature values
    
    Returns:
        Dictionary with prediction results
    """
    # Load model artifacts
    with open(model_path, 'rb') as f:
        artifacts = pickle.load(f)
    
    model = artifacts['model']
    scaler = artifacts['scaler']
    feature_columns = artifacts['feature_columns']
    target_labels = artifacts['target_labels']
    
    # Prepare input data
    if isinstance(input_features, dict):
        input_df = pd.DataFrame([input_features])
    else:
        input_df = input_features
    
    # Ensure all required features are present
    for col in feature_columns:
        if col not in input_df.columns:
            input_df[col] = 0  # Default value for missing features
    
    # Select and order features correctly
    X_input = input_df[feature_columns]
    
    # Scale features
    X_input_scaled = scaler.transform(X_input)
    
    # Make predictions
    predictions = model.predict(X_input)
    probabilities = model.predict_proba(X_input)
    
    results = []
    for i in range(len(predictions)):
        result = {
            'predicted_class': int(predictions[i]),
            'predicted_label': target_labels[predictions[i]],
            'probabilities': {
                'Normal': float(probabilities[i][0]),
                'Suspect': float(probabilities[i][1]),
                'Pathological': float(probabilities[i][2])
            },
            'confidence': float(max(probabilities[i]))
        }
        results.append(result)
    
    return results if len(results) > 1 else results[0]

# Test the prediction function
test_sample = X_test.iloc[0:1]
prediction_result = predict_fetal_health(model_filename, test_sample)

print("\n=== Test Prediction Function ===")
print(f"Prediction result: {prediction_result}")
print(f"Actual label: {health_labels_reverse[y_test.iloc[0]]}")

## Summary and Conclusions

In [ ]:
print("\n" + "="*50)
print("FETAL HEALTH CLASSIFICATION MODEL SUMMARY")
print("="*50)

print(f"\nDataset Size: {df.shape[0]} samples, {df.shape[1]-1} features")
print(f"Target Classes: 3 (Normal: {(df['fetal_health']==1).sum()}, Suspect: {(df['fetal_health']==2).sum()}, Pathological: {(df['fetal_health']==3).sum()})")

print(f"\nBest Model: XGBoost Classifier")
print(f"Best Parameters: {grid_search.best_params_}")

print(f"\nFinal Test Performance:")
print(f"  - Accuracy: {test_metrics['accuracy']:.4f}")
print(f"  - Precision: {test_metrics['precision']:.4f}")
print(f"  - Recall: {test_metrics['recall']:.4f}")
print(f"  - F1-Score: {test_metrics['f1']:.4f}")

print(f"\nTop 5 Most Important Features:")
for i, (idx, row) in enumerate(feature_importance.head(5).iterrows()):
    print(f"  {i+1}. {row['feature']}: {row['importance']:.4f}")

print(f"\nModel Files Created:")
print(f"  - {model_filename} (Complete model with preprocessing)")
print(f"  - fetal_health_xgboost.json (XGBoost model only)")

print(f"\nModel is ready for deployment and real-time fetal health monitoring!")
print("="*50)